In [ ]:
import re
import pandas as pd
from PyPDF2 import PdfReader

<strong>Ekstraksi PDF UNAIR IS</strong>

In [ ]:
# 1. Baca semua teks dari PDF
def read_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"
    return full_text

# 2. Ekstrak seluruh blok matkul
def extract_all_courses(text):
    blocks = re.split(r"Module designation\s+", text, flags=re.IGNORECASE)
    data = []

    for block in blocks[1:]:  # blok pertama sebelum matkul pertama di-skip
        lines = block.strip().splitlines()
        nama_matkul = lines[0].strip()

        # Capaian Pembelajaran
        capaian_match = re.search(
            r"(?:objectives/intended\s+learning outcomes|Learning Outcomes)\s+(.*?)\n(?:Content|This course discusses)",
            block,
            re.IGNORECASE | re.DOTALL
        )
        capaian = capaian_match.group(1).strip().replace('\n', ' ') if capaian_match else ""

        # Konten Mata Kuliah
        konten_match = re.search(
            r"(?:Content|This course discusses)\s*[:\-]?\s*(.*?)(?:\nExamination forms|\nFinal score|\nStudy and examination)",
            block,
            re.IGNORECASE | re.DOTALL
        )
        konten = konten_match.group(1).strip().replace('\n', ' ') if konten_match else ""

        data.append({
            "Course Title": nama_matkul,
            "Learning Outcomes": capaian,
            "Course Description": "",  # dikosongkan
            "Course Content": konten
        })
    
    return data

# 3. Eksekusi
pdf_path = r"C:\Users\asus\Downloads\wlewle\Capaian Pembelajaran\UNAIR_IS.pdf"  # pastikan path benar
text = read_pdf_text(pdf_path)
data = extract_all_courses(text)

# 4. Simpan ke Excel
df = pd.DataFrame(data)
df.to_excel(r"C:\Users\asus\Downloads\wlewle\Data\UNAIR_IS.xlsx", index=False)

print(f"✅ Berhasil menyimpan ke 'UNAIR_IS.xlsx'")

✅ Berhasil menyimpan ke 'UNAIR_IS.xlsx'


<strong>Ekstraksi PDF BINUS CS</strong>

In [ ]:
# Fungsi untuk membaca semua teks dari file PDF
def read_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"
    return full_text

# Fungsi untuk mengekstrak blok per mata kuliah
def extract_all_courses(text):
    # Pecah berdasarkan kata "Learning Outcomes"
    blocks = re.split(r"\n(?=[A-Z0-9 &/().-]{5,}\n?Learning Outcomes\s*:?)", text, flags=re.IGNORECASE)

    data = []

    for block in blocks:
        # Ambil nama matkul (biasanya baris pertama)
        lines = block.strip().splitlines()
        if not lines or len(lines[0]) < 5:
            continue  # Lewati blok kosong atau tidak valid

        title = lines[0].strip().upper()

        # Ambil Learning Outcomes
        lo_match = re.search(r"Learning Outcomes\s*:?\s*(.+?)(?:Topics\s*:|TOPICS\s*:)", block, re.DOTALL | re.IGNORECASE)
        lo = re.sub(r"\s+", " ", lo_match.group(1)).strip() if lo_match else ""

        # Ambil Topics/Konten
        topics_match = re.search(r"(?:Topics|TOPICS)\s*:?\s*(.+)", block, re.DOTALL | re.IGNORECASE)
        topics = re.sub(r"\s+", " ", topics_match.group(1)).strip() if topics_match else ""

        # Masukkan ke dalam struktur data
        data.append({
            "Module Name": title,
            "Learning Outcomes": lo,
            "Content": topics
        })

    return data

# Jalankan proses
pdf_path = r"C:\Users\asus\Downloads\wlewle\Capaian Pembelajaran\BINUS_CS.pdf"  # Pastikan file ada di direktori yang sama
text = read_pdf_text(pdf_path)
data = extract_all_courses(text)

# Simpan ke Excel
df = pd.DataFrame(data)
df.to_excel(r"C:\Users\asus\Downloads\wlewle\Data\BINUS_CS.xlsx", index=False)

print(f"✅ Berhasil menyimpan ke BINUS_CS.xlsx")


✅ Berhasil menyimpan ke BINUS_CS.xlsx


<strong>Ekstraksi PDF UNHAS CE</strong>

In [ ]:
# Fungsi membaca semua teks dari PDF
def read_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

# Fungsi mengekstrak modul per mata kuliah
def extract_courses(text):
    course_blocks = re.findall(
        r"Module name\s+(.*?)\s+Module level.*?Module.*?objectives/intended\s+learning\s+outcomes.*?"
        r"(?:(?:Intended\s+)?Learning Outcomes.*?)?(?:Course Learning Objective.*?)?"
        r"(.*?)\s+Content\s+Students will learn about\s*:\s*(.*?)(?:Forms of\s+Assessment|Assessment techniques)",
        text,
        re.DOTALL | re.IGNORECASE,
    )

    data = []
    for course in course_blocks:
        title = course[0].strip()
        outcomes = re.sub(r"\s+", " ", course[1]).strip()
        content = re.sub(r"\s+", " ", course[2]).strip()
        data.append({
            "Module Name": title,
            "Learning Outcomes": outcomes,
            "Content": content
        })
    return data

# Jalankan proses ekstraksi dan simpan ke Excel
pdf_path = r"C:\Users\asus\Downloads\wlewle\Capaian Pembelajaran\UNHAS_CE.pdf"  # Ubah sesuai path lokal
text = read_pdf_text(pdf_path)
courses = extract_courses(text)

# Simpan ke Excel
df = pd.DataFrame(courses)
df.to_excel(r"C:\Users\asus\Downloads\wlewle\Data\UNHAS_CE.xlsx", index=False)

print("✅ Berhasil menyimpan ke 'UNHAS_CE.xlsx'")


✅ Berhasil menyimpan ke 'UNHAS_CE.xlsx'


<strong>Ekstraksi PDF UGM CS</strong>

In [ ]:
def read_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

def extract_all_module_data(text):
    # --- Ekstrak semua module names dengan re.findall ---
    module_names = re.findall(
        r"Module name\s*([^\n]{3,100}?)\s*(?:\n|Module level|Code|Courses|Semester|Undergraduate)",
        text, re.IGNORECASE
    )

    # --- Ekstrak blok per modul untuk ambil LO dan Content ---
    blocks = re.split(r"\n?Module name\s+", text)
    modules = []

    for i, block in enumerate(blocks[1:], 1):  # lewati blok pertama (pra-modul)
        # Module Name diambil dari hasil findall yang terpisah
        module_name = module_names[i-1].strip() if i-1 < len(module_names) else "UNKNOWN"

        # Learning Outcomes
        lo_match = re.search(
            r"(?:Learning outcomes.*?(?:PLOs|corresponding PLOs))\s*(.*?)\s*(?:Content|Study and)",
            block, re.DOTALL | re.IGNORECASE
        )
        learning_outcomes = lo_match.group(1).strip().replace("\n", " ") if lo_match else ""

        # Content
        content_match = re.search(
            r"Content\s*(.*?)\s*(?:Study and|Assessment|Reading list|Examination)",
            block, re.DOTALL | re.IGNORECASE
        )
        content = content_match.group(1).strip().replace("\n", " ") if content_match else ""

        modules.append({
            "Module Name": module_name,
            "Learning Outcomes": learning_outcomes,
            "Content": content
        })

    return modules

# --- Jalankan ---
pdf_path = r"C:\Users\asus\Downloads\wlewle\Capaian Pembelajaran\UGM_CS.pdf"
output_path = r"C:\Users\asus\Downloads\wlewle\Data\UGM_CS.xlsx"

text = read_pdf_text(pdf_path)
modules_data = extract_all_module_data(text)

df = pd.DataFrame(modules_data)
df.to_excel(output_path, index=False)
print(f"✅ Berhasil menyimpan ke: {output_path}")

Multiple definitions in dictionary at byte 0x7d for key /Subtype
Multiple definitions in dictionary at byte 0x7d for key /Subtype


✅ Berhasil menyimpan ke: C:\Users\asus\Downloads\wlewle\Data_CPL\UGM_CS.xlsx


<strong>Ekstraksi PDF ITS CE</strong>

In [ ]:
import pdfplumber

# Daftar kata yang ingin dihapus dari Deskripsi dan LO
WORDS_TO_REMOVE = [
    r"\bCourse\b", r"\bDescription\b", r"\(Deskripsi", r"\bMata\b", r"\bKuliah\)",
    r"\bLearning\b", r"\boutcomes\b", r"\band\b", r"\btheir\b", r"\bcorresponding\b", r"\bPLOs\b",
    r"\(Capaian", r"\bPembelajaran\b", r"\bKuliah\b", r"\)"
]

def clean_text(text):
    for word in WORDS_TO_REMOVE:
        text = re.sub(word, "", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

# Baca semua teks dari pdf
def read_pdf_with_pdfplumber(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

# Ekstraksi isi mata kuliah
def extract_courses(text):
    # Pecah per mata kuliah berdasarkan heading 'Course (Mata Kuliah)'
    blocks = re.split(r"\nCourse\s*\(Mata Kuliah\)", text, flags=re.IGNORECASE)
    data = []

    for i, block in enumerate(blocks[1:], start=1):  # Lewati preambule
        course = {}

        # FIXED Course Title
        title_match = re.search(r"^(.*?)\s+Code", block)
        if title_match:
            course["Module Name"] = title_match.group(1).strip()
        else:
            lines = block.strip().split("\n")
            first_valid_line = next((line for line in lines if line.strip()), None)
            course["Module Name"] = first_valid_line.strip() if first_valid_line else f"Unknown Title {i}"

        # Course Description
        desc_match = re.search(
            r"(Course\s*Description\s*\(.*?\).{0,2000}?)(?=\n(?:Learning outcomes|Language|Prerequisite|Content|Study and examination|Media employed|Reading list|Course\s*\(Mata Kuliah\)|\Z))",
            block, re.DOTALL)
        course["Description"] = clean_text(desc_match.group(1)) if desc_match else ""

        # Learning Outcomes
        lo_match = re.search(
            r"(Learning outcomes.*?\(.*?\).{0,2000}?)(?=\n(?:Content|Study and examination|Media employed|Reading list|Course\s*\(Mata Kuliah\)|\Z))",
            block, re.DOTALL)
        course["Learning Outcomes"] = clean_text(lo_match.group(1)) if lo_match else ""

        # Course Content
        content_match = re.search(
            r"Content\s*\(.*?\)\s*(.*?)(?=\n(?:Study and examination|Media employed|Reading list|Course\s*\(Mata Kuliah\)|\Z))",
            block, re.DOTALL)
        course["Content"] = re.sub(r"\s+", " ", content_match.group(1)).strip() if content_match else ""

        data.append(course)

    return data

# Eksekusi
pdf_path = r"C:\Users\asus\Downloads\wlewle\Capaian Pembelajaran\ITS_CE.pdf" 
text = read_pdf_with_pdfplumber(pdf_path)
courses = extract_courses(text)

# Simpan
df = pd.DataFrame(courses)
df.to_excel("ITS_CE_B.ind.xlsx", index=False)
print("✅ Berhasil menyimpan ke 'ITS_CE_B.ind.xlsx'")


✅ Berhasil menyimpan ke 'ITS_CE_B.ind.xlsx'


In [ ]:
# 1. Baca file Excel hasil ekstraksi
file_path = r"C:\Users\asus\Downloads\wlewle\ITS_CE_B.ind.xlsx"
df = pd.read_excel(file_path)

# 2. Daftar kata-kata umum Bahasa Indonesia
INDO_WORDS = [
    "adalah", "dengan", "dan", "yang", "pada", "oleh", "untuk", "dari", "mata kuliah ini",
    "menjelaskan", "menggunakan", "sebagai", "dapat", "akan", "mampu", "diharapkan",
    "merupakan", "konsep", "pemahaman", "pelajaran", "memberikan", "terhadap", "mahasiswa",
    "topik", "tujuan", "bertujuan", "hasil", "belajar", "kuliah", "ilmu", "dasar", "melalui",
    "memahami", "materi", "mempelajari", "pembelajaran", "teori", "praktik", "contoh", "studi",
    "secara", "dalam", "ini", "lain", "lingkup", "cakupan", "pengantar", "khusus",
    "umum", "kompetensi", "pembahasan", "tentang", "sistem", "kerja", "peran", "fungsi", "jenis",
    "level", "struktur", "elemen", "faktor", "meliputi", "pendekatan", "mengembangkan",
    "ditujukan", "diperkenalkan", "aktivitas", "berbasis", "diharuskan", "dipelajari"
]

# 3. Kata-kata khas Bahasa Inggris
ENGLISH_HINTS = [
    "the", "an", "a", "students", "learn", "understand", "will", "can", "must",
    "introduction", "language", "software", "hardware", "system", "process", "project", "course",
    "is", "are", "of", "in", "to", "data", "network", "information", "method", "program"
]

# 4. Fungsi mendeteksi apakah baris lebih ke arah Bahasa Indonesia
def is_indonesian(line):
    if not isinstance(line, str):
        return False
    indo_score = sum(1 for word in INDO_WORDS if re.search(rf"\b{word}\b", line, re.IGNORECASE))
    eng_score = sum(1 for word in ENGLISH_HINTS if re.search(rf"\b{word}\b", line, re.IGNORECASE))
    return indo_score > eng_score  # Hanya anggap Indonesia jika skornya lebih tinggi

# 5. Fungsi membersihkan kalimat Bahasa Indonesia
def remove_indonesian_sentences(text):
    if not isinstance(text, str):
        return text
    lines = re.split(r'(?<=[.?!])\s+|\n', text)  # pisah kalimat dan newline (ALT+ENTER)
    english_lines = [line.strip() for line in lines if not is_indonesian(line)]
    return " ".join(english_lines).strip() if english_lines else text  # fallback ke teks asli jika semua terhapus

# 6. Bersihkan kolom Description, LO, Content
columns_to_clean = ["Description", "Learning Outcomes", "Content"]
for col in columns_to_clean:
    if col in df.columns:
        df[col] = df[col].apply(remove_indonesian_sentences)

# 7. Bersihkan kolom Module Name dari isi dalam kurung (termasuk jika tidak ditutup)
if "Module Name" in df.columns:
    df["Module Name"] = df["Module Name"].apply(
        lambda x: re.sub(r"\s*\(.*$", "", x).strip() if isinstance(x, str) else x
    )

# 8. Simpan hasil
output_path = r"C:\Users\asus\Downloads\wlewle\Data\ITS_CE.xlsx"
df.to_excel(output_path, index=False)
print(f"✅ File hasil telah disimpan: {output_path}")

✅ File hasil telah disimpan: C:\Users\asus\Downloads\wlewle\Data_CPL\ITS_CE.xlsx


<strong>Ekstraksi PDF ITS IS</strong>

In [ ]:
import pdfplumber
import re
import pandas as pd

# Baca semua teks menjadi satu string panjang
pdf_path = r"C:\Users\asus\Downloads\wlewle\Capaian Pembelajaran\ITS_IS.pdf" 
with pdfplumber.open(pdf_path) as pdf:
    full_text = "\n".join([p.extract_text() for p in pdf.pages if p.extract_text()])

# Bersihkan spasi tab dll
full_text = re.sub(r'[ \t]+', ' ', full_text)

# Pola regex untuk mengambil semua bagian dari satu matkul
pattern = re.compile(
    r'Matakuliah\s*\n\s*(.*?)\s*'                                # Module Name
    r'Kode\s*:\s*\w+.*?'                                         # Skip kode/SKS
    r'Deskripsi Matakuliah\s*(.*?)\s*'                           # Description
    r'Capaian Pembelajaran Matakuliah\s*(.*?)\s*'                # Learning Outcomes
    r'Tujuan Pembelajaran yang Spesifik\s*.*?'                   # Skip tujuan
    r'Pokok Bahasan\s*(.*?)\s*Pustaka Utama',                    # Content (BETWEEN Tujuan & Pustaka)
    re.DOTALL
)

matches = pattern.findall(full_text)

# Simpan hasilnya
data = []
for m in matches:
    data.append({
        "Module Name": m[0].strip(),
        "Description": m[1].strip(),
        "Learning Outcomes": m[2].strip(),
        "Content": m[3].strip()
    })

# Export ke Excel
df = pd.DataFrame(data)
df = df.drop_duplicates(subset="Module Name")
df.to_excel("ITS_IS_Indo.xlsx", index=False)


Cannot set gray non-stroke color because /'P21' is an invalid float value


In [3]:
%pip install openpyxl deep-translator

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
import time

# Baca file Excel
file_path = r"C:\Users\asus\Downloads\wlewle\ninuninu\ITS_IS_Indo.xlsx"  # Ubah jika file di path berbeda
df = pd.read_excel(file_path)

# Kolom-kolom yang akan diterjemahkan
kolom_terjemahkan = ["Module Name", "Description", "Learning Outcomes", "Content"]

# Fungsi untuk menerjemahkan teks
def translate_text(text):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return text
    try:
        translated = GoogleTranslator(source='id', target='en').translate(text)
        time.sleep(1)  # jeda untuk menghindari batas kuota
        return translated
    except Exception as e:
        return f"[Translation error] {text}"

# Proses translate dan overwrite
for kolom in kolom_terjemahkan:
    if kolom in df.columns:
        df[kolom] = df[kolom].apply(translate_text)
    else:
        print(f"Kolom '{kolom}' tidak ditemukan.")

# Simpan ke file baru (hasil full dalam Bahasa Inggris)
df.to_excel(r"C:\Users\asus\Downloads\wlewle\Data\ITS_IS.xlsx", index=False)
print("✅ File terjemahan berhasil disimpan sebagai ITS_IS.xlsx")


✅ File terjemahan berhasil disimpan sebagai ITS_IS.xlsx
